In [2]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
cm_number = 10
cm = plt.cm.get_cmap("tab10")
colors = [cm(1. / (cm_number - 1) * i) for i in range(0, cm_number)]
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset, zoomed_inset_axes
mpl.rc('font', family='Arial')
mpl.rc('font',size = 8)
mpl.rc('mathtext',fontset = 'stix')
mpl.rc('xtick', labelsize=6)
mpl.rc('ytick', labelsize=6)
# mpl.rc('xtick', labelsize=4)
# mpl.rc('ytick', labelsize=4)
mpl.rc('axes', labelsize=8)
mpl.rc('axes', labelpad=1)
mpl.rc('axes', titlesize=8)
mpl.rc('axes', linewidth=0.5)
figsavepath = "../../Elements/MicroBeam"

C:\Users\TengMa\AppData\Local\Temp\ipykernel_5332\1995258559.py:7: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cm = plt.cm.get_cmap("tab10")


In [3]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
datadir = r'C:\Users\TengMa\OneDrive\POLIMI\1. MEMS SINDy\wetransfer_data_perseus_2025-05-16_1416\data_perseus'
positionname = 'sindy_perseus_S_FRF_IC0.mat'
velocityname = 'sindy_perseus_S_vel_FRF_IC0.mat'
datafile = "%s/%s"%(datadir, positionname)
P = "%s/%s"%(datadir, 'sindy_perseus_I_FRF_IC0.mat')
POM = "%s/%s"%(datadir, 'POM.mat')
import h5py
with h5py.File(datafile, 'r') as file:
    data = np.array(file['S'])
with h5py.File(P, 'r') as file:
    p = np.array(file['I'])


In [4]:
from scipy.io import loadmat
timestamp = 99999
timeeffect = 99999
Modes = 3
Modeshape = loadmat(POM)["POM"]
node = 7178
POM_1 = Modeshape[node,:3]
print(POM_1)

[ 0.04229482  0.02260849 -0.01295628]


In [5]:
import importlib
import sys
sys.path.append("../")
import EvLOWN
importlib.reload(EvLOWN)  # 重新载入
library = [
    lambda x:x[0],
    lambda x:x[0]*x[0],
    lambda x:x[0]*x[0]*x[0],
    lambda x:x[1],
    lambda x:x[1]*x[1],
    lambda x:x[1]*x[1]*x[1],
    lambda x:x[2],
    lambda x:x[2]*x[2],
    lambda x:x[2]*x[2]*x[2],
    lambda x:x[3],
    lambda x:x[3]*x[3],
    lambda x:x[3]*x[3]*x[3],
    lambda x:x[4],
    lambda x:x[4]*x[4],
    lambda x:x[4]*x[4]*x[4],
    lambda x:x[5],
    lambda x:x[5]*x[5],
    lambda x:x[5]*x[5]*x[5],

]
library_name = [
    lambda x:x[0],
    lambda x:x[0]+x[0],
    lambda x:x[0]+x[0]+x[0],
    lambda x:x[0]+x[0]+x[0]+x[0],
    lambda x:x[0]+x[0]+x[0]+x[0]+x[0],
    lambda x:x[1],
    lambda x:x[1]+x[1],
    lambda x:x[1]+x[1]+x[1],
    lambda x:x[1]+x[1]+x[1]+x[1],
    lambda x:x[1]+x[1]+x[1]+x[1]+x[1],
    lambda x:x[0]+x[1],
    lambda x:x[0]+x[0]+x[1],
    lambda x:x[0]+x[1]+x[1],
    lambda x:x[0]+x[0]+x[0]+x[1],
    lambda x:x[0]+x[0]+x[1]+x[1],
    lambda x:x[0]+x[1]+x[1]+x[1],
    lambda x:x[0]+x[0]+x[0]+x[0]+x[1],
    lambda x:x[0]+x[0]+x[0]+x[1]+x[1],
    lambda x:x[0]+x[0]+x[1]+x[1]+x[1],
    lambda x:x[0]+x[1]+x[1]+x[1]+x[1],
]
dim = 3
model_EvLOWN = EvLOWN.WeakNOForce(dim,library,library_name)

In [137]:
# Loading Datas

num = [6,10,21]
paras = []
times = []
Forces = []
Displacements = []
Velocitys = []
Modes = 3
for i in range(len(num)):
    
    start = num[i]*timestamp+1
    end = (num[i]+1)*timestamp
    F_omega1 = p[1,start]
    F_amplitude = p[2,start]
    paras.append((F_omega1, F_amplitude))
    t = p[0,start:start+timeeffect]
    times.append(t)
    Forces.append(F_amplitude*np.cos(F_omega1*t))
    displacement = data[:Modes,start:start+timeeffect]
    velocity = np.zeros([Modes,timeeffect])
    for mode in range(Modes):
        velocity[mode,:] = np.gradient(data[mode,start:start+timeeffect],t)
    Displacements.append(displacement)
    Velocitys.append(velocity)


In [138]:
# Get Evolutionary Variables
Dots = []
Phis = []
Frequencys = []
for i in range(len(num)):
    model_EvLOWN.Get_frequency(Displacements[i].T,Velocitys[i].T,times[i])
    Frequencys.append(model_EvLOWN.frequencys[0])
    model_EvLOWN.frequencys[1] = model_EvLOWN.frequencys[0]
    model_EvLOWN.frequencys[2] = model_EvLOWN.frequencys[0]
    model_EvLOWN.Get_Evolution(order = [0,1,2])
    model_EvLOWN.Library_rebuild(Forces[i])
    Dots.append(model_EvLOWN.dot)
    Phis.append(model_EvLOWN.Phi)
    

In [139]:
# Pre-processing
import copy
New_dots = copy.deepcopy(Dots)
Omegaidxs = [0,6,12]
mean_frequency = np.mean(Frequencys)
for i in range(len(num)):
    for j in range(1):
        for m in range(5):
            New_dots[i][j][:,m] += (Frequencys[i]**2-mean_frequency**2)*Phis[i][j][:,m,Omegaidxs[j]]


In [140]:
from EvLOWN import cyxpy_solver
stp = 10
enp = 10
orders = [[1,2],[0,3,4],[0,3,4]]
Xis = np.zeros([len(library)+1,Modes])
for i in range(Modes):
    y = []
    X = []
    for j in range(len(num)):
        y.append(New_dots[j][i][stp:-enp,:])
        X.append(Phis[j][i][stp:-enp,:,:])
    y = np.concatenate(y,axis = 0)
    X = np.concatenate(X,axis = 0)
    Xi_i = cyxpy_solver(y, X, orders[i], lam=0.1, stop_tolerance=0.05)
    Xis[:,i] = Xi_i
    Xis[Omegaidxs[i],i] += mean_frequency**2
    # Xis[-1,d] = Xis[-1,i]

start:8.31062
Round 0, step: 0.45585, now: 0.54415, selected: 0
Round 1, step: 0.10961, now: 0.43455, selected: 2
Round 2, step: 0.13254, now: 0.30200, selected: 18
Round 3, step: 0.27558, now: 0.02643, selected: 3


C:\Users\TengMa\anaconda3\Lib\site-packages\cvxpy\problems\problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


start:433.35883
Round 0, step: 0.90981, now: 0.09019, selected: 4
Round 1, step: 0.00778, now: 0.08241, selected: 13
Round 2, step: 0.05460, now: 0.02781, selected: 10
start:396.92245
Round 0, step: 0.07243, now: 0.92757, selected: 1
Round 1, step: 0.92086, now: 0.00672, selected: 4


In [141]:
print(Xis)

[[ 3.38319826e-02  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00 -9.18654785e-07]
 [-1.18890727e-11  0.00000000e+00  0.00000000e+00]
 [ 1.84615607e-04  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  3.70061812e-05  2.37689179e-05]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  3.37018516e-02  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -2.96961407e-03  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  3.37018516e-02]
 [ 0.00000000e+00  2.29684505e-04  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 1.06443375e-01  0.00000000e+00  0.00000000e+00]]


In [142]:
import util
importlib.reload(util)  # 重新载入
library_matlab = [
    'u1','u1^2','u1^3','v1','v1^2','v1^3',
    'u2','u2^2','u2^3','v2','v2^2','v2^3',
    'u3','u3^2','u3^3','v3','v3^2','v3^3',
    'A*c'
]
util.generate_beam_from_library(
    func_name = "Mirror_POM3_4",
    out_path = "../Matcont_solver/models/MicroBeam/Mirror_POM3_6,10,21.m",
    n_dof = Modes,
    library = library_matlab,
    coef = -1*np.array(Xis),
    tol = 1e-12
    
)
# util.append_coef_only_csv(
#     csv_path = "Beam_coef_2.csv",
#     library = library_matlab,
#     coef = -1*np.array(Xis),
#     eq_names = ['v1','v2','v3'],
#     idx = num,
#     tol = 1e-13
# )


'..\\Matcont_solver\\models\\MicroBeam\\Mirror_POM3_6,10,21.m'

In [143]:
import os
import numpy as np
import matlab.engine
from tqdm import tqdm

PROJECT = r"../matcont_solver/"   # 改成你的路径
RESULTS = os.path.join(PROJECT, "results")
os.makedirs(RESULTS, exist_ok=True)

models = ["Mirror_POM3_6,10,21"]  # 你可以放多个模型函数名
As = [0.5,0.75,1.0,1.25,1.50]

omega0 = 0.1839
Qvalue = 1000.0
X0 = [0,0,0,0,0,0,1,0]

eng = matlab.engine.start_matlab()
eng.cd(PROJECT, nargout=0)
eng.addpath(eng.genpath(PROJECT), nargout=0)
print(eng.pwd())
print(eng.which('run_frf_mirror'))

for i in range(1,2):
    model = "Mirror_POM3_6,10,21"
    for A in tqdm(As, desc=f"A sweep ({model})", position=1, leave=False):
        eng.eval(f"global Global_A; Global_A = {A};", nargout=0)
        
        out = eng.run_frf_mirror(
            model,
            float(omega0*1.1),
            matlab.double(X0),   # MATLAB 内部会 X0(:)
            float(Qvalue),
            nargout=1
        )
        # print(model, A, "is_empty:", eng.isempty(out))
        eng.workspace["xlcc"] = out
        savename = os.path.join(RESULTS, f"{model}_A_{A:.5f}.mat")
        eng.save(savename, "xlcc", nargout=0)

eng.quit()

C:\Users\TengMa\OneDrive\文章\9. MEMS_FEM_EvLOWN\Notebook\Matcont_solver
C:\Users\TengMa\OneDrive\文章\9. MEMS_FEM_EvLOWN\Notebook\Matcont_solver\run_frf_mirror.m



A sweep (Mirror_POM3_6,10,21): 100%|████████████████████████████████████████████████████| 5/5 [16:45<00:00, 216.51s/it]
                                                                                                                       

In [144]:
def xlcc2frf(xlcc,Modeshape, node):
    d1, d2 = np.shape(xlcc)
    num = 3
    orbit_len = (d1-2-2-num*2)//(num*2+2)
    frfs = np.zeros(d2)
    for j in range(d2):
        orbits = np.zeros([orbit_len,num])
        for i in range(num):
            orbits[:,i] = xlcc[i*2:-((num+1)*2)-2+i*2:(num+1)*2, j]
        orbit_e2 = np.sum(orbits * Modeshape[node-1,:3], axis = 1)
        orbit_e3 = np.sum(orbits * Modeshape[node,:3], axis = 1)
        frfs_orbit =  2*np.degrees(np.arctan2(np.sqrt(orbit_e2**2 + orbit_e3**2)/2,950))
        frfs[j] = np.max(frfs_orbit)
    return xlcc[-1,:], frfs

In [7]:
# 计算ROM的预测结果
import pandas as pd
Omegas_FOMs = []
response_FOMs = []
Omegas_ROMs = []
response_ROMs = []
node = 7178
POM_1 = Modeshape[node,:3]
# PROJECT = r"../matcont_solver/"   # 改成你的路径
As = [0.5,0.75,1.0,1.25,1.50]
model_mirror = "Mirror_POM3_6——21"
betas = [1.0,1.5,2.0,2.5,3.0]
trainset = pd.read_csv("../../Validation/Micromirror2/trainset.csv")
# Train_FREQ = trainset['omegas'][num]
# Train_FRF = trainset['As'][num]
# model_mirror = 'Mirror_POM3_%d'%num
for i in range(len(betas)):
    file = "../../Validation/Micromirror2/beta%.1f-1.csv"%betas[i]
    data_FOM = pd.read_csv(file, header = None)
    Omegas_FOMs.append(data_FOM[0])
    response_FOMs.append(data_FOM[1])
    # if i >=0:
    #     y = scipy.io.loadmat('../../Validation/Micromirror2/matcont_origin_norm_3_%.2f_%d.mat'%(betas[i]/2,num))
    #     y = y['xlcc']
    #     freq, frf = xlcc2frf(y, Modeshape, 4526)

    
    #     frf = frf*1000
    # else:
    y = scipy.io.loadmat('%s/results/%s_A_%.5f'%(PROJECT,model_mirror,betas[i]/2))
    y = y['xlcc']
# print(POM_1)
    freq, frf = xlcc2frf(y, Modeshape, 4526)
    Omegas_ROMs.append(freq[:-1])
    response_ROMs.append(frf[:-1])
        

NameError: name 'scipy' is not defined

In [ ]:
fig, ax = plt.subplots(figsize = (9/2.54, 6.5/2.54),dpi = 300)
cm_number = 8
cm = plt.cm.get_cmap("Blues")
colors = [cm(1. / (cm_number - 1) * i) for i in range(0, cm_number)]
length = 1000
l_idx = [0,51,56,61,65]
r_idx = [0,51,59,65,70]
for i in range(len(betas)):
    
    plt.plot(Omegas_FOMs[i], response_FOMs[i], color = colors[i+3],alpha = 0.5,lw = 1, ls = '-')
    plt.plot(Omegas_ROMs[i], response_ROMs[i], lw = 0.75,ls = '--', color = colors[i+3])
# plt.plot(Omegas_ROM, np.degrees(np.arcsin(response_ROM[0,:]/length)),lw = 0.75, ls = '--', color = colors[i+3])
    # if i<1:

    #     plt.plot(Omegas_ROM, np.degrees(np.arcsin(response_ROM[i,:]/length)),lw = 0.75, ls = '--', color = colors[i+3])
    # else:
    #     plt.plot(Omegas_ROM[:l_idx[i]], np.degrees(np.arcsin(response_ROM[i,:l_idx[i]]/length)),lw = 0.75, ls = '--', color = colors[i+3])
    #     plt.plot(Omegas_ROM[r_idx[i]:], np.degrees(np.arcsin(response_ROM[i,r_idx[i]:]/length)),lw = 0.75, ls = '--', color = colors[i+3])
# plt.plot(freq[:-5],frf[:-5])
ax.scatter([Train_FREQ],[Train_FRF], color = 'red', s = 4,zorder = 100,label = 'Training Point')
ax.plot([0,0],[1,1],ls = '-',alpha = 0.5,lw = 1,color = 'black',label = 'Ground Truth')
ax.plot([0,0],[1,1],ls = '--',alpha = 1,lw = 0.75,color = 'black',label = 'Prediction')
plt.legend(fontsize = 6)
# plt.scatter(Omegas_FOM, responses_FOM, s =9,marker = '*',label = 'FOM',zorder = 100)
ax.set_ylim(0,12.5)
ax.tick_params(direction='out',width = 0.5,length = 1)
ax.set_xlim(0.1832,0.1844)
ax.set_xlabel('Frequency $\Omega$ (Hz)')
ax.set_ylabel(r'$\theta$ (deg)')

cmap_discrete = mpl.colors.ListedColormap(colors[3:])
bounds = list(range(3, cm_number+1))
norm = mpl.colors.BoundaryNorm(bounds, cmap_discrete.N)

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap_discrete)
sm.set_array([])
tick_positions = [3.5, 4.5, 5.5, 6.5, 7.5]
# 你可以根据实际意义随意改 label 文本
tick_labels = ['1.0$\mu$N','1.5$\mu$N','2.0$\mu$N','2.5$\mu$N','3.0$\mu$N']  
ax.tick_params(direction='out',width = 0.5,length = 1)
cbar = fig.colorbar(sm, ax=ax, ticks=tick_positions)
cbar.ax.set_yticklabels(tick_labels)
cbar.set_label(r"Forcing intensity $\beta$",labelpad = 10)
cbar.ax.minorticks_off()
cbar.ax.tick_params(direction='out',width = 0.5,length = 1)

In [349]:
Omegas_ROM

[array([0.63249582, 0.63222716, 0.6318805 , 0.63143416, 0.63085842,
        0.63012195, 0.62918426, 0.62799743, 0.62650652, 0.62465125,
        0.62236984, 0.61960461, 0.61631178, 0.61247382, 0.60811044,
        0.60329236, 0.59814994, 0.59286412, 0.58766694, 0.58328852,
        0.58010851, 0.57778716, 0.5761178 , 0.57494475, 0.57417099,
        0.57372642, 0.5735619 , 0.57363864, 0.57392736, 0.57440451,
        0.5750512 , 0.57585213, 0.57679452, 0.57786767, 0.57906163,
        0.58036922, 0.58178356, 0.58329621, 0.58490377, 0.58660126,
        0.58837823, 0.59023584, 0.5921701 , 0.59417602, 0.59625097,
        0.5983913 , 0.60059395, 0.60285444, 0.60516865, 0.6075327 ,
        0.60994042, 0.61238483, 0.61485663, 0.61733889, 0.61979751,
        0.62211283, 0.6225966 , 0.62276986, 0.62277762, 0.62276035,
        0.62271976, 0.62270679, 0.62268916, 0.62267523, 0.62266079,
        0.62264335, 0.62262364, 0.62260157, 0.62257662, 0.62254796,
        0.62251463, 0.62247539, 0.62242875, 0.62

In [196]:
num = int(len(p[2,:])/timestamp)
Omegas_FOM = []
Amplitudes_FOM = []
# responses_FOM = np.zeros(num)
AFs = [0.0938,0.125,0.1562,0.1875,0.2188,0.25,0.2812,0.3125,0.375,0.5,0.625,0.75]
for i in range(len(AFs)):
    file = "../../Validation/MicroBeam/F_%.4f.csv"%AFs[i]
    data_FOM = pd.read_csv(file, header = None)
    Omegas_FOM.append(data_FOM[0])
    Amplitudes_FOM.append(data_FOM[1])

In [194]:
Amplitudes_FOM[0].shape

(1260001,)

In [195]:
data

,0,1
0,0.520176,1.515081
1,0.521935,1.570766
2,0.523518,1.633411
3,0.524837,1.696056
4,0.526684,1.786543
...,...,...
129,0.581566,1.403712
130,0.582885,1.334107
131,0.584908,1.250580
132,0.586667,1.187935
